<a href="https://colab.research.google.com/github/aishanikar9/BWSI_Operations_Team/blob/main/ResnetModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils
from PIL import Image
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision

In [ ]:
!wget -O RescueNet-classification-train.csv https://raw.githubusercontent.com/aishanikar9/BWSI_Operations_Team/main/RescueNet-classification-train.csv
!wget -O RescueNet-classification-val.csv https://raw.githubusercontent.com/aishanikar9/BWSI_Operations_Team/main/RescueNet-classification-val.csv
!wget -O RescueNet-classification-test.csv https://raw.githubusercontent.com/aishanikar9/BWSI_Operations_Team/main/RescueNet-classification-test.csv

--2026-07-27 18:16:58--  https://raw.githubusercontent.com/aishanikar9/BWSI_Operations_Team/main/RescueNet-classification-train.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 46761 (46K) [text/plain]
Saving to: ‘RescueNet-classification-train.csv’

RescueNet-classific 100%[===================>]  45.67K  --.-KB/s    in 0.001s  

2026-07-27 18:16:59 (53.7 MB/s) - ‘RescueNet-classification-train.csv’ saved [46761/46761]

--2026-07-27 18:16:59--  https://raw.githubusercontent.com/aishanikar9/BWSI_Operations_Team/main/RescueNet-classification-val.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443.

In [ ]:
import pandas as pd

train_df = pd.read_csv("RescueNet-classification-train.csv")
val_df = pd.read_csv("RescueNet-classification-val.csv")
test_df = pd.read_csv("RescueNet-classification-test.csv")

print(train_df.head())
print(val_df.head())
print(test_df.head())

    Image_ID  Neighborhood_ID
0  10930.jpg                2
1  11351.jpg                1
2  14254.jpg                0
3  15013.jpg                1
4  12373.jpg                1
    Image_ID  Neighborhood_ID
0  11251.jpg                2
1  11122.jpg                2
2  10997.jpg                2
3  11005.jpg                2
4  10925.jpg                2
    Image_ID  Neighborhood_ID
0  13363.jpg                1
1  11038.jpg                2
2  11965.jpg                2
3  13702.jpg                1
4  14376.jpg                1


In [ ]:
net = torchvision.models.resnet50(weights = 'IMAGENET1K_V2')

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 196MB/s]


In [ ]:
net.fc = nn.Linear(2048, 3)
net = nn.DataParallel(net)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net.to(device)
print("ResNet ready")

ResNet ready


In [ ]:
count = train_df["Neighborhood_ID"].value_counts().sort_index()
class_weights = torch.tensor((count.sum() / (len(count)* count)).values, dtype=torch.float).to(device)

#weighing different classes

In [ ]:
for parameters in net.parameters():
  parameters.requires_grad =False

In [ ]:
# Loss function, optimizer, scheduler
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(net.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, 0.9)

In [ ]:
#transform

In [ ]:
#equivalent to class LadiDataset(Dataset):


In [ ]:
#train_dataset and val_dataset will come from the RescueNetDataset() class
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False, num_workers=2)

NameError: name 'train_dataset' is not defined

In [ ]:
from torch.utils.tensorboard import SummaryWriter

In [ ]:
#train

def train_model(net, train_loader, test_loader, criterion, optimizer, scheduler, logs_path, model_name,
                starting_epoch=0, additional_epochs=30, print_every_num_batches=50):
  model_name_base = f'Resnet50-{model_name}' + '.ep{}.pth'
  writer = SummaryWriter(logs_path)
  checkpoints_path = logs_path/'checkpoints'
  checkpoints_path.mkdir(parents=True, exist_ok=True)

  if starting_epoch > 0:
    load_path = checkpoints_path / model_name_base.format(str(starting_epoch).zfill(3))
    net.load_state_dict(torch.load(load_path, map_location=device))

  #adding mixed precision, less memory, higher speed
  use_amp = torch.cuda.is_available()
  scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

  #tracking the best model instead of just the first
  best_value_f1 = -1.0

  for epoch in range(starting_epoch, starting_epoch+additional_epochs):
    net.train()
    running_loss = 0.0
    running_epoch_loss = 0.0

    for i, data in enumerate(train_loader,0):
      inputs = data['image'].to(device)
      labels = data['label'].to(device)

      optimizer.zero_grad()

      #temporarily cuts bits in half, doubling run time and halfing GPU memory,
      with torch.autocast(device_type='cuda' if use_amp else 'cpu', enabled=use_amp):
        outputs = net(inputs)
        loss = criterion(outputs, labels)

      #multiplies loss when it get's really small so the model keeps learning, reversed by .backward() before weights update tho
      scaler.scale(loss).backward()

      #unscale, then update the weights, checks for infinite and NAN first tho
      scaler.step(optimizer)

      #update the scale amount before the next loop
      scaler.update()

      running_loss += loss.item()
      running_epoch_loss += loss.item()

      if (i + 1) % print_every_num_batches == 0:
        print(f'[epoch {epoch+1}, batch {i+1}] average loss: {running_loss/print_every_num_batches:.4f}')
        running_loss = 0.0


    average_epoch_loss = running_epoch_loss/(i+1)
    writer.add_scalar('Loss/epoch_avg/train', average_epoch_loss, epoch)
    print(f'[epoch {epoch+1}] average training epoch loss: {average_epoch_loss}')
    writer.add_scalar('LR/rate', scheduler.get_last_lr()[0], epoch)
    print(f'[epoch {epoch+1}] average training loss: {average_epoch_loss:.4f}')
    scheduler.step()



    #validation time:
    net.eval()
    val_running = 0.0
    with torch.no_grad(): #basically tells it to stop doing gradient computation
      for i, data in enumerate(test_loader, 0):
        inputs = data['image'].to(device)
        labels = data['label'].to(device)
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        val_running += loss.item()

      val_loss = val_running / (i + 1)
      print(f'[epoch {epoch+1}] val loss: {val_loss:.4f}')
      epoch_string = str(epoch + 1).zfill(3)
      torch.save(net.state_dict(), checkpoints_path / model_name_base.format(epoch_string)) #saves the model state as a .pth file every epoch

  writer.close()

In [ ]:
#Evaluate